In [1]:
dic = {
'PASC001':'14741',
'PASC003':'14744',
'PASC005':'14926',
'PASC009':'14829',
'PASC010':'s10016a',
'PASC011':'14983',
'PASC012':'14938',
'PASC013':'14824',
'PASC014':'14785',
'PASC015':'14875',
'PASC018':'14922',
'PASC019':'14880',
'PASC020':'14995',
'PASC022':'s14919',
'PASC023':'14990',
'PASC024':'15005',
'PASC025':'15028',
'PASC027':'s15076',
'PASC029':'s15256',
'PASC030':'s15162',
'PASC033':'s15185',
'PASC034':'s15248',
'PASC035':'s15304',
'PASC037':'s15503',
'PASC038':'s15359',
'PASC039':'s15459',
'PASC040':'s15473'}

In [2]:
len(dic)

27

In [6]:
Control_list = [ "PASC001_cb_L_R_good", "PASC003_ck_R_L_good", "PASC004_tc", "PASC005_af_R_L_good", "PASC007_ee", "PASC009_rg_R_L_good", 
                "PASC010_bk_R_L_good", "PASC011_ls_R_L_good","PASC012_kk1_R_L_good","PASC013_va_L_R_good", "PASC014_sl_L_R_good",
                "PASC015_rj_R_L_good", "PASC018_mm_L_R_good", "PASC019_bo_L_R_good", "PASC020_lj_R_L_good", "PASC022_po_R_L_good",  
                "PASC023_bj_R_L_good","PASC024_ko_L_R_good", "PASC025_hr_L_R_good",  "PASC027_yj_R_L_good",
                "PASC028_kp_L_R_good", "PASC029_aa_L_R_good", "PASC030_kk2_L_R_good", "PASC032_pm", "PASC033_tb_L_R_good",
                "PASC034_mk_R_L_good", "PASC035_sk_R_L_good", "PASC036_ss", "PASC037_gr_L_R_good", "PASC038_ge_R_L_good", 
                "PASC039_fk_R_L_good",  "PASC040_tj_L_R_good", "PASC041_aa", "FASC004_tk_L"]

                 
#Control_list = ["PASC019_bo_L_R_good"]
# --done PASC019_bo_L_R_good
len(Control_list)

Control_list = [ "PASC004_tc"]

Control_list_without_MRI = ["PASC004_tc", "PASC007_ee", "PASC032_pm", "PASC036_ss", "PASC041_aa", 'PASC028_kp_L_R_good', "FASC004_tk_L"]


In [10]:
import mne
from mne.datasets import sample
from mne.datasets import eegbci
from mne.datasets import fetch_fsaverage
import os.path as op
import os

src_num = 4

# ==============================
# Create forward solution folder ONCE
# ==============================
forward_sol_folder = "/mnt/isilon/w_neuro/gopalarlab/Pain_project/data/PASC_MEG_data/Forward_files"
os.makedirs(forward_sol_folder, exist_ok=True)


for sub in Control_list:
    num = "2"
    if sub in ["PASC004_tc", "PASC007_ee", "PASC032_pm", "PASC036_ss", "PASC041_aa"]:
        #path = "/mnt/beegfs/malann/codes/Pain_project/data/PASC_MEG_data/Resting_state_only/" + sub
        path = f'/mnt/isilon/w_neuro/gopalarlab/PASC_MEG_data/Resting state only/{sub}'
    else:
        #path = "/mnt/beegfs/malann/codes/Pain_project/data/PASC_MEG_data/"+sub
        path = f"/mnt/isilon/w_neuro/gopalarlab/PASC_MEG_data/{sub}"

    if sub in ['PASC012_kk1_R_L_good','PASC030_kk2_L_R_good']:
        val = sub[7:11]
    else:
        val = sub[7:10]
    sub_name = sub[0:7]

    file_name = path+"/SPONT_"+num+val+"_raw_quat_tsss.fif"
    raw = mne.io.read_raw_fif(file_name)
    info = mne.io.read_info(file_name)
    
    if sub in Control_list_without_MRI:
            # sample FSaverage data dir
        fs_dir = fetch_fsaverage(verbose=True)
        subjects_dir = op.dirname(fs_dir)
        subject = "fsaverage"
        trans = "fsaverage"  # MNE has a built-in fsaverage transformation
        #src = op.join(fs_dir, "bem", "fsaverage-ico-5-src.fif")
        bem = op.join(fs_dir, "bem", "fsaverage-5120-5120-5120-bem-sol.fif")
        src = mne.setup_source_space(
            subject, spacing=f"ico{src_num}", add_dist="patch", subjects_dir=subjects_dir
        )
    else:
    
        # The paths to Freesurfer reconstructions
        mri_data_path = "/mnt/beegfs/malann/codes/Pain_project/data/Freesurfer_Files"
        subjects_dir = mri_data_path+'/subject'+dic[sub_name]
        subject = "sample"
        
        trans = path+ "/audvis_raw-trans.fif"
        src = mne.setup_source_space(
            subject, spacing=f"ico{src_num}", add_dist="patch", subjects_dir=subjects_dir
        )

        conductivity = (0.3,)  # for single layer
        # conductivity = (0.3, 0.006, 0.3)  # for three layers
        model = mne.make_bem_model(
            subject=subject, ico=4, conductivity=conductivity, subjects_dir=subjects_dir
        )
        bem = mne.make_bem_solution(model)

    
    fwd = mne.make_forward_solution(
        raw.info,
        trans=trans,
        src=src,
        bem=bem,
        meg=True,
        eeg=False,
        mindist=5.0,
        n_jobs=None,
        verbose=True,
    )

    

    # ==============================
    # Save forward
    # ==============================
    forward_sol = f"{forward_sol_folder}/{sub_name}_SPONT_2_ico{src_num}_raw-fwd.fif"

    mne.write_forward_solution(forward_sol, fwd, overwrite=True)

    
    #forward_sol = f"/mnt/isilon/w_neuro/gopalarlab/PASC_MEG_data/data/PASC_MEG_data/Forward_files/{sub_name}_SPONT_2_ico{src_num}_raw-fwd.fif"
    
    mne.write_forward_solution(forward_sol, fwd, overwrite=True, verbose=None)

Opening raw data file /mnt/isilon/w_neuro/gopalarlab/PASC_MEG_data/Resting state only/PASC004_tc/SPONT_2_tc_raw_quat_tsss.fif...
    Range : 4000 ... 1419999 =      0.800 ...   284.000 secs
Ready.
Opening raw data file /mnt/isilon/w_neuro/gopalarlab/PASC_MEG_data/Resting state only/PASC004_tc/SPONT_2_tc_raw_quat_tsss-1.fif...
    Range : 1420000 ... 1507999 =    284.000 ...   301.600 secs
Ready.
0 files missing from root.txt in /home/malann/mne_data/MNE-fsaverage-data
0 files missing from bem.txt in /home/malann/mne_data/MNE-fsaverage-data/fsaverage
Setting up the source space with the following parameters:

SUBJECTS_DIR = /home/malann/mne_data/MNE-fsaverage-data
Subject      = fsaverage
Surface      = white
Icosahedron subdivision grade 4


/tmp/ipykernel_715661/1557001125.py:33: RuntimeWarning: This filename (/mnt/isilon/w_neuro/gopalarlab/PASC_MEG_data/Resting state only/PASC004_tc/SPONT_2_tc_raw_quat_tsss.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(file_name)



>>> 1. Creating the source space...

Doing the icosahedral vertex picking...
Loading /home/malann/mne_data/MNE-fsaverage-data/fsaverage/surf/lh.white...
Mapping lh fsaverage -> ico (4) ...
    Triangle neighbors and vertex normals...
Loading geometry from /home/malann/mne_data/MNE-fsaverage-data/fsaverage/surf/lh.sphere...
Setting up the triangulation for the decimated surface...
loaded lh.white 2562/163842 selected to source space (ico = 4)

Loading /home/malann/mne_data/MNE-fsaverage-data/fsaverage/surf/rh.white...
Mapping rh fsaverage -> ico (4) ...
    Triangle neighbors and vertex normals...
Loading geometry from /home/malann/mne_data/MNE-fsaverage-data/fsaverage/surf/rh.sphere...
Setting up the triangulation for the decimated surface...
loaded rh.white 2562/163842 selected to source space (ico = 4)

Calculating patch information (limit=0.0 mm)...
    Computing patch statistics...
    Patch information added...
    Computing patch statistics...
    Patch information added...
You 